# Step 05 — Build a Single HMM Estimation Panel

**Purpose:** Create one trip-level Parquet dataset for the household-year latent-state / trip-level-emission HMM, using the validated Step 03, 03B, and 04 files. Each trip repeats its household-year covariates and the same scalar 2007 initialization value.

**Inputs (local restricted data):** `hmm_trips`, `hmm_annual_covariates_income_imputed.parquet`, `household_demographics_with_events_2008_2017.tsv`, and `hmm_initial_fresh_fv_share_2007.parquet`.

**Outputs (local restricted data only):** `processed/hmm_input/hmm_estimation_panel.parquet`, plus small validation CSVs under `processed/hmm_input/validation/`.

**Do not commit outputs to GitHub.** Preserve household/trip IDs as strings. Never interpret missing life-event flags as zero. No prediction split is applied.

**Research convention:** 2007 pooled fresh-FV dollar share is a **starting value** for a common initial Healthy-state probability, not an observed latent-state probability or a fixed final parameter.
**TSV parsing note:** Demographic rows include additional columns beyond the four age/join fields; DuckDB reads the complete tab-separated schema before selecting these fields.


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
MASTER_DIR = DATA_ROOT / "processed/hmm_panel/hmm_master"
TRIP_GLOB = MASTER_DIR / "hmm_trips/panel_year=*/trips.parquet"
ANNUAL_FILE = MASTER_DIR / "hmm_annual_covariates_income_imputed.parquet"
DEMOGRAPHICS_FILE = DATA_ROOT / "processed/demographics/household_demographics_with_events_2008_2017.tsv"
INITIAL_FILE = MASTER_DIR / "hmm_initial_fresh_fv_share_2007.parquet"
OUTPUT_DIR = DATA_ROOT / "processed/hmm_input"
AUDIT_DIR = OUTPUT_DIR / "validation"
OUTPUT_FILE = OUTPUT_DIR / "hmm_estimation_panel.parquet"
TEMP_FILE = OUTPUT_DIR / "hmm_estimation_panel.parquet.partial"
OVERWRITE_EXISTING = False  # Set True only to explicitly replace an existing complete panel.

MEMORY_LIMIT = "4GB"
THREADS = 4

for path in [ANNUAL_FILE, DEMOGRAPHICS_FILE, INITIAL_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")
if not list((MASTER_DIR / "hmm_trips").glob("panel_year=*/trips.parquet")):
    raise FileNotFoundError(f"No yearly trip parquet files found: {TRIP_GLOB}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "temp").mkdir(parents=True, exist_ok=True)
if OUTPUT_FILE.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(f"Output exists. Set OVERWRITE_EXISTING=True to rebuild: {OUTPUT_FILE}")

def sql_path(path):
    """Quote a filesystem path for a DuckDB SQL string literal."""
    return "'" + str(path).replace(chr(92), "/").replace("'", "''") + "'"

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute(f"SET temp_directory = {sql_path(DATA_ROOT / 'temp')}")

con.execute(f"""
    CREATE VIEW trips AS
    SELECT * FROM read_parquet({sql_path(TRIP_GLOB)}, hive_partitioning=true, union_by_name=true)
""")
con.execute(f"CREATE VIEW annual AS SELECT * FROM read_parquet({sql_path(ANNUAL_FILE)})")
con.execute(f"CREATE VIEW init AS SELECT * FROM read_parquet({sql_path(INITIAL_FILE)})")
# The demographic TSV contains many columns. Do not pass a four-column
# `columns={...}` schema to read_csv: that describes the entire file and
# causes "Expected Number of Columns: 4" when real rows contain more.
# Parse the complete TSV schema, then project only the four fields needed.
# Explicit empty quote/escape settings prevent accidental quote autodetection.
con.execute(f"""
    CREATE VIEW demo AS
    SELECT
        CAST(hhid AS VARCHAR) AS hhid,
        TRY_CAST(panel_year AS INTEGER) AS panel_year,
        TRY_CAST(male_head_age_from_birth AS DOUBLE) AS male_age,
        TRY_CAST(female_head_age_from_birth AS DOUBLE) AS female_age
    FROM read_csv({sql_path(DEMOGRAPHICS_FILE)},
                  delim='\t', header=true, all_varchar=true,
                  quote='', escape='', strict_mode=true)
""")
print("Input views created successfully.")

## Preflight checks

Ensure that one annual row and one demographic row exist per `(hhid, panel_year)`, that the common 2007 share is a single valid scalar, and that needed columns are present. The 2008–2017 trip universe is inherited from Step 03; no rows are added from trips outside the six-category union.

In [ ]:
EMISSIONS = ["fv_y", "milk_y", "yogurt_y", "cereal_y", "snack_y", "beverage_y"]
EVENTS = ["event_new_child", "getting_married", "getting_divorce_or_separation",
          "moving", "getting_job", "retirement", "other_unemployment"]
SPENDING = [
    "fresh_spending", "non_fresh_spending",
    "skimmed_milk_spending", "low_fat_milk_spending", "other_milk_spending",
    "skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending",
    "low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending",
    "healthy_snack_spending", "unhealthy_snack_spending",
    "healthy_beverage_spending", "unhealthy_beverage_spending"
]
TRIP_COLUMNS = ["hhid", "panel_year", "trip_id", "purchase_date", *EMISSIONS, *SPENDING, "food_spend"]
ANNUAL_COLUMNS = ["hhid", "panel_year", *EVENTS, "K_1", "income10k_imputed", "family_size_numeric"]

def check_columns(view, required):
    existing = {r[1] for r in con.execute(f"PRAGMA table_info('{view}')").fetchall()}
    missing = sorted(set(required) - existing)
    if missing:
        raise ValueError(f"{view} is missing required columns: {missing}")

check_columns("trips", TRIP_COLUMNS)
check_columns("annual", ANNUAL_COLUMNS)
check_columns("init", ["initial_fresh_fv_share_2007"])

for view in ["annual", "demo"]:
    duplicates = con.execute(f"""
        SELECT hhid, panel_year, COUNT(*) AS n
        FROM {view} GROUP BY hhid, panel_year HAVING COUNT(*) > 1 LIMIT 5
    """).df()
    if not duplicates.empty:
        raise ValueError(f"Duplicate household-year keys in {view}:\n{duplicates}")

initial_summary = con.execute("""
    SELECT COUNT(*) AS n, MIN(initial_fresh_fv_share_2007) AS min_pi,
           MAX(initial_fresh_fv_share_2007) AS max_pi
    FROM init
""").fetchone()
if initial_summary[0] != 1 or initial_summary[1] is None or not (0 < initial_summary[1] < 1):
    raise ValueError(f"Expected exactly one initial 2007 share in (0,1), got {initial_summary}")
print(f"Common initial FV share: {initial_summary[1]:.8f}")

# Check age coverage in HMM annual household-years, NOT the broader demographic source.
age_stats = con.execute("""
    SELECT COUNT(*) AS annual_rows,
           COUNT(d.hhid) AS age_source_matches,
           COUNT(*) FILTER (WHERE d.male_age IS NULL AND d.female_age IS NULL) AS missing_age,
           COUNT(*) FILTER (WHERE COALESCE(d.male_age, d.female_age) < 18
                              OR COALESCE(d.male_age, d.female_age) > 110) AS suspicious_head_age
    FROM annual a LEFT JOIN demo d
      ON CAST(a.hhid AS VARCHAR)=d.hhid AND CAST(a.panel_year AS INTEGER)=d.panel_year
""").df()
print(age_stats.to_string(index=False))
if int(age_stats.loc[0, "missing_age"]) > 0:
    raise ValueError("Age missing in the HMM annual sample; investigate before building Step 05.")

## Build the one-file HMM estimation panel

The resulting output has **one row per shopping trip**, with 15 category-specific spending variables, total `food_spend`, the requested annual demographic controls and events, and the common initial share. No `income_missing_indicator`, Regular/Magnet split, or estimation index columns are included. Age is the mean of available male/female head ages derived from birth years; if only one exists, use that head's age.

In [ ]:
if TEMP_FILE.exists():
    TEMP_FILE.unlink()  # Remove only our own incomplete Step 05 output.

# An inner join makes matching explicit; any dropped trip rows are audited below.
# Importantly, missing annual life events remain NULL, not 0.
selected_emissions = ",\n        ".join(
    f"TRY_CAST(t.{name} AS TINYINT) AS {name}" for name in EMISSIONS
)
selected_events = ",\n        ".join(
    f"TRY_CAST(a.{name} AS TINYINT) AS {name}" for name in EVENTS
)
selected_spending = ",\n        ".join(
    f"TRY_CAST(t.{name} AS DOUBLE) AS {name}" for name in SPENDING
)

build_sql = f"""
    SELECT
        CAST(t.hhid AS VARCHAR) AS hhid,
        CAST(t.panel_year AS INTEGER) AS panel_year,
        CAST(t.trip_id AS VARCHAR) AS trip_id,
        CAST(t.purchase_date AS DATE) AS purchase_date,
        {selected_emissions},
        {selected_events},
        TRY_CAST(a.K_1 AS TINYINT) AS K_1,
        TRY_CAST(a.income10k_imputed AS DOUBLE) AS income10k_imputed,
        TRY_CAST(a.family_size_numeric AS DOUBLE) AS family_size_numeric,
        CASE
            WHEN d.male_age IS NOT NULL AND d.female_age IS NOT NULL
                THEN (d.male_age + d.female_age) / 2.0
            ELSE COALESCE(d.male_age, d.female_age)
        END AS age,
        {selected_spending},
        TRY_CAST(t.food_spend AS DOUBLE) AS food_spend,
        CAST(i.initial_fresh_fv_share_2007 AS DOUBLE) AS initial_fresh_fv_share_2007
    FROM trips t
    INNER JOIN annual a
      ON CAST(t.hhid AS VARCHAR)=CAST(a.hhid AS VARCHAR)
     AND CAST(t.panel_year AS INTEGER)=CAST(a.panel_year AS INTEGER)
    INNER JOIN demo d
      ON CAST(t.hhid AS VARCHAR)=d.hhid
     AND CAST(t.panel_year AS INTEGER)=d.panel_year
    CROSS JOIN init i
"""

# Stream query results straight to a single compressed Parquet file; no giant pandas DataFrame.
con.execute(f"COPY ({build_sql}) TO {sql_path(TEMP_FILE)} (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f"Built temporary output: {TEMP_FILE}")

## Post-build verification and publish

Calculate row/household coverage, outcome validity, missing event counts, and spending and age sanity checks. Fail before publishing if a required invariant is violated. The **trip key** is `(hhid, panel_year, trip_id, purchase_date)`. Reporting audits are kept outside GitHub.

In [ ]:
con.execute(f"CREATE VIEW output_check AS SELECT * FROM read_parquet({sql_path(TEMP_FILE)})")
source_counts = con.execute("""
    SELECT COUNT(*) AS source_trips,
           COUNT(DISTINCT (CAST(hhid AS VARCHAR), CAST(panel_year AS INTEGER),
                           CAST(trip_id AS VARCHAR), CAST(purchase_date AS DATE))) AS unique_source_trip_keys
    FROM trips
""").df()
output_counts = con.execute("""
    SELECT COUNT(*) AS output_trips, COUNT(DISTINCT hhid) AS output_households,
           COUNT(DISTINCT (hhid, panel_year)) AS output_household_years,
           COUNT(DISTINCT (hhid, panel_year, trip_id, purchase_date)) AS unique_output_trip_keys,
           COUNT(*) FILTER (WHERE age IS NULL) AS missing_age,
           COUNT(*) FILTER (WHERE income10k_imputed IS NULL) AS missing_income,
           COUNT(*) FILTER (WHERE food_spend < 0) AS negative_food_spend,
           COUNT(*) FILTER (WHERE initial_fresh_fv_share_2007 IS NULL) AS missing_initial_share
    FROM output_check
""").df()
summary = pd.concat([source_counts, output_counts], axis=1)
print(summary.to_string(index=False))
summary.to_csv(AUDIT_DIR / "step05_sample_summary.csv", index=False)

n_source = int(source_counts.loc[0,"source_trips"])
n_out = int(output_counts.loc[0,"output_trips"])
if int(source_counts.loc[0,"unique_source_trip_keys"]) != n_source:
    raise ValueError("Step 03 source contains duplicate trip keys.")
if n_out != n_source:
    raise ValueError(f"Output row count {n_out:,} differs from trip source {n_source:,}; check annual/demographic joins.")
if int(output_counts.loc[0,"unique_output_trip_keys"]) != n_out:
    raise ValueError("Duplicate trip keys in merged output.")
if int(output_counts.loc[0,"missing_age"]) or int(output_counts.loc[0,"missing_income"]):
    raise ValueError("Age/income missing after merge.")
if int(output_counts.loc[0,"negative_food_spend"]):
    raise ValueError("Negative total food spending detected.")
if int(output_counts.loc[0,"missing_initial_share"]):
    raise ValueError("Common initial share missing.")

# Five non-cereal categories use 0/1/2; cereal uses 0/1/2/3.
validations = []
for name in EMISSIONS:
    max_allowed = 3 if name == "cereal_y" else 2
    wrong = con.execute(f"""
        SELECT COUNT(*) FROM output_check
        WHERE {name} IS NULL OR {name} NOT BETWEEN 0 AND {max_allowed}
    """).fetchone()[0]
    validations.append({"variable": name, "invalid_or_missing": wrong})
emission_audit = pd.DataFrame(validations)
emission_audit.to_csv(AUDIT_DIR / "step05_emission_validation.csv", index=False)
print("\nEmission validity:\n", emission_audit.to_string(index=False))
if (emission_audit.invalid_or_missing > 0).any():
    raise ValueError("Invalid or missing emission code(s); see emission audit.")

# Event missingness is an informational audit: never silently change NA to 0.
event_audit = con.execute("SELECT " + ", ".join(
    f"COUNT(*) FILTER (WHERE {name} IS NULL) AS {name}" for name in EVENTS
) + " FROM output_check").df().T.reset_index()
event_audit.columns = ["event", "missing_trip_rows"]
event_audit.to_csv(AUDIT_DIR / "step05_event_missingness.csv", index=False)
print("\nMissing life-event trip rows:\n", event_audit.to_string(index=False))

if OUTPUT_FILE.exists():
    if not OVERWRITE_EXISTING:
        raise FileExistsError(OUTPUT_FILE)
    OUTPUT_FILE.unlink()
TEMP_FILE.replace(OUTPUT_FILE)
print(f"\nSaved FINAL single Parquet: {OUTPUT_FILE}")
print(f"Number of variables: {len(con.execute('DESCRIBE output_check').fetchall())}")
print("Model input is ready for estimation after review of the validation CSVs.")

In [ ]:
# Optional: inspect the final schema without reading the full file into memory.
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet({sql_path(OUTPUT_FILE)})").df().to_string(index=False))
con.close()